In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense , Input , LSTM , Dropout , BatchNormalization , Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
df = pd.read_csv("/content/english_to_bangla.csv" , )
# df = df.iloc[:-1 , :]
df.head()

,en,bn
0,a child in a pink dress is climbing up a set o...,একটি গোলাপী জামা পরা বাচ্চা মেয়ে একটি বাড়ির প্...
1,a girl going into a wooden building .,একটি মেয়ে শিশু একটি কাঠের বাড়িতে ঢুকছে
2,a little girl climbing into a wooden playhouse .,একটি বাচ্চা তার কাঠের খেলাঘরে উঠছে ।
3,a little girl climbing the stairs to her playh...,ছোট মেয়েটি তার খেলার ঘরের সিড়ি বেয়ে উঠছে
4,a little girl in a pink dress going into a woo...,গোলাপি জামা পড়া ছোট একটি মেয়ে একটি কাঠের তৈরি...


In [4]:
df.isna().sum()

,0
en,0
bn,0


In [20]:
x_tokenizer = Tokenizer()
y_tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')

In [21]:
x = df['en']
y = "<start> " + df['bn'] + " <end>"

x_tokenizer.fit_on_texts(x)
y_tokenizer.fit_on_texts(y)

In [22]:
y[0]

'<start> একটি গোলাপী জামা পরা বাচ্চা মেয়ে একটি বাড়ির প্রবেশ পথের সিঁড়ি বেয়ে উঠছে। <end>'

In [24]:
for word , index in y_tokenizer.word_index.items():
  if word == "<end>":
    print(word , index)
    break

<end> 2


In [25]:
x = x_tokenizer.texts_to_sequences(x)
y = y_tokenizer.texts_to_sequences(y)

In [32]:
def find_eng_word(index):
  for word , ind in x_tokenizer.word_index.items():
    if(ind == index):
      return word

def find_ben_word(index):
  for word , ind in y_tokenizer.word_index.items():
    if(ind == index):
      return word

In [33]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.2 , random_state = 1)

In [34]:
print(len(x_train))
print(len(y_train))

31252
31252


In [35]:
eng_str = ""

for ind in x_train[0]:
  eng_str = eng_str + " " + find_eng_word(ind)

print(eng_str)


ben_str = ""

for ind in y_train[0]:
  ben_str = ben_str + " " + find_ben_word(ind)

print(ben_str)

 a boy in a hat on a swing
 <start> টুপি পরা এক ছেলে দোলনায় <end>


In [44]:
no_of_lstm_cell = 128
embedding_dim = 256

def build_encoder_decoder_model( input_vocab_size , output_vocab_size):

  # encoder model
  encoder_input = Input(shape=(None,))
  encoder_emb = Embedding(input_vocab_size , embedding_dim ,  mask_zero=True)(encoder_input)

  encoder_lstm_layer = LSTM(no_of_lstm_cell , return_state=True)
  encoder_lstm_output , encoder_h_state , encoder_c_state = encoder_lstm_layer(encoder_emb)

  encoder_state = [encoder_h_state , encoder_c_state]

  # decoder
  decoder_input = Input(shape=(None ,))
  decoder_emb = Embedding(output_vocab_size , embedding_dim ,  mask_zero=True)(decoder_input)

  decoder_lstm_layer = LSTM(no_of_lstm_cell , return_sequences=True  )
  decoder_lstm_output = decoder_lstm_layer(decoder_emb , initial_state=encoder_state)

  softmax_layer = Dense(output_vocab_size , activation='softmax')
  decoder_output = softmax_layer(decoder_lstm_output)

  model = Model([encoder_input , decoder_input] , decoder_output)

  return model


In [45]:
encoder_input = x_train
decoder_input = [row[:-1] for row in y_train]
decoder_output = [row[1:] for row in y_train]

# For encoder input
encoder_input = pad_sequences(encoder_input, padding='post')  # pad with 0 at the end

# For decoder input
decoder_input = pad_sequences(decoder_input, padding='post')

# For decoder target
decoder_output = pad_sequences(decoder_output, padding='post')

In [46]:
model = build_encoder_decoder_model(len(x_tokenizer.word_index)+1 , len(y_tokenizer.word_index)+1 )
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if target is one-hot
    metrics=['accuracy']
)

In [47]:
decoder_output

array([[  76,   21,    6, ...,    0,    0,    0],
       [   3,   12,   27, ...,    0,    0,    0],
       [   3, 1122,   38, ...,    0,    0,    0],
       ...,
       [ 806, 1678,  357, ...,    0,    0,    0],
       [   3, 3400, 2261, ...,    0,    0,    0],
       [  16,   22,   29, ...,    0,    0,    0]], dtype=int32)

In [49]:
history = model.fit(
    [encoder_input , decoder_input],
    decoder_output,
    batch_size = 32 ,
    epochs = 5,
    validation_split = 0.2
)

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 52s 67ms/step - accuracy: 0.1500 - loss: 2.2415 - val_accuracy: 0.1121 - val_loss: 3.8682
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 53s 68ms/step - accuracy: 0.1559 - loss: 2.0958 - val_accuracy: 0.1116 - val_loss: 3.9045
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 65ms/step - accuracy: 0.1641 - loss: 1.9407 - val_accuracy: 0.1113 - val_loss: 3.9430
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step - accuracy: 0.1718 - loss: 1.8063 - val_accuracy: 0.1106 - val_loss: 3.9977
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 50s 64ms/step - accuracy: 0.1783 - loss: 1.7001 - val_accuracy: 0.1105 - val_loss: 4.0436


In [50]:
x_test = pad_sequences(x_test , padding='post')

predictions = model.predict(x_test)

y_true = pad_sequences(y_test, padding='post')

# predictions = predictions.flatten()
# y_true = y_true.flatten()

# r2 = r2_score(y_true, predictions)
# print(f"R² score: {r2}")

ValueError: Layer "functional_1" expects 2 input(s), but it received 1 input tensors. Inputs received: [<tf.Tensor 'data:0' shape=(32, 28) dtype=int32>]

In [ ]:
predictions